# Stage 6: how very old people walk (HAR70+), and why the models call it "light"

**The question.** In Stage 5, older people's (60–81) normal and fast walking was often called *light* instead of *moderate-vigorous*. Is their walking really gentler and slower, or are the models wrong?

**HAR70+** has 18 people aged 70 to 95 (the dataset's description; no age file is included), recorded at 50 Hz while going about semi-structured daily activities. It has **no wrist sensor**, only the **lower back** and **right thigh**, so our wrist models can't be tested on it. But the **rhythm of walking** (steps per minute) is the same everywhere on the body, so it can be measured here and compared.

1. Read HAR70+ and cut the walking into 10 s windows
2. Measure cadence (steps per minute) two independent ways, back and thigh, and check they agree
3. How fast, how regular, and how intense is very old people's walking?
4. How long are their walking bouts? (Stage 4 found the HMM hides short bursts)
5. Back to GOTOV: measure cadence from its chest sensor during walking, and see whether the windows the models called "light" really were slower

**No model is trained here.** **Kernel:** `.venv-tf` in this folder. Run Stage 5 first.

## How this works (for beginners)

**Cadence** = steps per minute. A step is one foot landing; a **stride** is two steps (left + right).

**Why cadence matters.** Research on adults up to age 85 (the CADENCE-Adults studies) puts the line for *moderate intensity* (about 3 MET, the same line Capture-24 uses for moderate-vigorous) at about **100 steps per minute**. So cadence gives an intensity check that doesn't depend on where the sensor is worn.

**Finding the cadence in a signal.** Walking repeats, so the signal repeats. The **frequency spectrum** (Fourier transform) shows how strongly each rhythm is present, and the strongest one in the walking range is the walking rhythm:

| Sensor | What repeats | Its rhythm | Cadence |
|---|---|---|---|
| Lower back (trunk) | a jolt every time *either* foot lands | the **step** rhythm | rhythm × 60 |
| Thigh | the leg swings forward and back once per stride | the **stride** rhythm | rhythm × 120 |

Two different sensors, two different rhythms. If they give the same cadence, the method can be trusted.

**Regularity.** Compare the signal with itself shifted by one step (*autocorrelation*). 1 = every step identical, 0 = no pattern. A shift of two steps (one full stride) compares each leg with itself. Unsteady or shuffling gait scores lower.

In [1]:
import glob
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from common import CLASSES, RESULTS_DIR, paired_bootstrap

HAR_DIR = Path('../../data/open datasets/HAR 70')
GOTOV_DIR = Path('../../data/open datasets/GOTOV Human Physical Activity and Energy Expenditure Dataset on Older Individuals_2_all')
HZ = 50
WIN_S = 10                                         # WHY 10 s: long enough for 10+ steps, short enough to fit most bouts
LABELS = {1: 'walking', 3: 'shuffling', 4: 'stairs up', 5: 'stairs down', 6: 'standing', 7: 'sitting', 8: 'lying'}
MODERATE_CADENCE = 100                             # steps/min, about 3 MET (CADENCE-Adults)
files = sorted(HAR_DIR.glob('*.csv'))
print(len(files), 'people:', [f.stem for f in files])

18 people: ['501', '502', '503', '504', '505', '506', '507', '508', '509', '510', '511', '512', '513', '514', '515', '516', '517', '518']


## 1. Read HAR70+ and cut the walking into windows

A **bout** is one unbroken stretch of the same activity (no gap over 0.1 s). Each walking or shuffling bout is cut into 10 s windows from its start; the leftover end is dropped.

In [2]:
def load(f):
    d = pd.read_csv(f)
    t = pd.to_datetime(d['timestamp']).to_numpy('datetime64[ms]').astype('int64') / 1000   # seconds
    return t, d[['back_x', 'back_y', 'back_z']].to_numpy(), d[['thigh_x', 'thigh_y', 'thigh_z']].to_numpy(), d['label'].to_numpy()

def bouts(labels, t):
    """(start, end, label) of every unbroken stretch of one activity."""
    change = np.flatnonzero((labels[1:] != labels[:-1]) | (np.diff(t) > 0.1)) + 1
    edges = np.r_[0, change, len(labels)]
    return [(a, b, labels[a]) for a, b in zip(edges[:-1], edges[1:])]

data, all_bouts = {}, []
for f in files:
    t, back, thigh, labels = load(f)
    assert set(np.unique(labels)) <= set(LABELS), np.unique(labels)
    assert abs(np.median(np.diff(t)) - 1 / HZ) < 1e-3       # 50 Hz
    data[f.stem] = (t, back, thigh, labels)
    all_bouts += [{'pid': f.stem, 'activity': LABELS[lab], 'start': a, 'end': b, 'seconds': (b - a) / HZ}
                  for a, b, lab in bouts(labels, t)]
all_bouts = pd.DataFrame(all_bouts)
print('minutes per activity, all 18 people:')
print((all_bouts.groupby('activity').seconds.sum() / 60).round(1).sort_values(ascending=False).to_string())

minutes per activity, all 18 people:
activity
walking        359.8
sitting        161.2
standing       139.4
lying           67.7
shuffling       22.0
stairs down      1.7
stairs up        1.5


## 2. Cadence, measured two ways

- **Back:** the strongest rhythm between 1 and 3 Hz (60–180 steps/min) in the *magnitude* of the back sensor (how hard the trunk is jolted, whichever way the sensor faces)
- **Thigh:** the strongest rhythm between 0.4 and 1.5 Hz in the thigh's front-back axis (`thigh_z`; it points down when sitting, so it points forwards when standing), times 2

Each window is windowed (Hann) and padded before the Fourier transform, so the rhythm can be read more finely than the 0.1 Hz steps of a plain 10 s transform.

In [3]:
def strongest_rhythm(signal, fs, low, high):
    """Frequency (Hz) of the strongest repeating rhythm between low and high."""
    s = (signal - signal.mean()) * np.hanning(len(signal))
    n_fft = 2 ** int(np.ceil(np.log2(len(s) * 16)))           # WHY pad: read the peak position finely
    power = np.abs(np.fft.rfft(s, n_fft)) ** 2
    freqs = np.fft.rfftfreq(n_fft, 1 / fs)
    band = (freqs >= low) & (freqs <= high)
    return freqs[band][power[band].argmax()]

def cadence_trunk(xyz, fs):
    return 60 * strongest_rhythm(np.linalg.norm(xyz, axis=1), fs, 1.0, 3.0)

def regularity(xyz, fs, cadence):
    """Autocorrelation of the trunk signal one step and one stride later (best match within 10% of the expected lag)."""
    s = np.linalg.norm(xyz, axis=1); s = s - s.mean()
    acf = np.correlate(s, s, 'full')[len(s) - 1:] / (s @ s)
    step = fs * 60 / cadence
    best = lambda lag: acf[int(lag * 0.9):int(lag * 1.1) + 1].max()
    return best(step), best(2 * step)

# check on fake data: a trunk jolting 110 times a minute, a thigh swinging 55 times a minute
tt = np.arange(10 * HZ) / HZ
fake_trunk = np.column_stack([np.ones_like(tt) + 0.3 * np.sin(2 * np.pi * 110 / 60 * tt), 0 * tt, 0 * tt])
assert abs(cadence_trunk(fake_trunk, HZ) - 110) < 1
assert abs(120 * strongest_rhythm(np.sin(2 * np.pi * 55 / 60 * tt), HZ, 0.4, 1.5) - 110) < 1
assert regularity(fake_trunk, HZ, 110)[0] > 0.8
print('cadence checks passed')

rows = []
for b in all_bouts[all_bouts.activity.isin(['walking', 'shuffling'])].itertuples():
    t, back, thigh, _ = data[b.pid]
    for w in range(int(b.seconds // WIN_S)):
        a = b.start + w * WIN_S * HZ
        seg_back, seg_thigh = back[a:a + WIN_S * HZ], thigh[a:a + WIN_S * HZ]
        cad = cadence_trunk(seg_back, HZ)
        step_reg, stride_reg = regularity(seg_back, HZ, cad)
        rows.append({'pid': b.pid, 'activity': b.activity, 'bout seconds': b.seconds,
                     'cadence (back)': cad, 'cadence (thigh)': 120 * strongest_rhythm(seg_thigh[:, 2], HZ, 0.4, 1.5),
                     'step regularity': step_reg, 'stride regularity': stride_reg,
                     'trunk jolt (g)': np.linalg.norm(seg_back, axis=1).std()})
win = pd.DataFrame(rows)
walk = win[win.activity == 'walking']

gap = (walk['cadence (back)'] - walk['cadence (thigh)']).abs()
print(f'{len(walk):,} walking windows. Back and thigh cadence within 5 steps/min: {100 * (gap <= 5).mean():.0f}% of windows '
      f'(median difference {gap.median():.1f} steps/min)')
print(f'where they disagree, the back reads lower {100 * ((walk["cadence (back)"] < walk["cadence (thigh)"])[gap > 5]).mean():.0f}% of the time')

cadence checks passed


1,772 walking windows. Back and thigh cadence within 5 steps/min: 66% of windows (median difference 1.5 steps/min)
where they disagree, the back reads lower 74% of the time


**Which to trust?** Use only the windows where the two sensors agree (within 5 steps/min). Those windows have a clear, regular rhythm both sensors can see; the rest are turns, starts and stops, or uneven steps. The share that agrees is itself a measure of how steady the walking is.

In [4]:
walk = walk.assign(agree=gap <= 5)
steady = walk[walk.agree]
per_person = steady.groupby('pid').agg(windows=('agree', 'size'), cadence=('cadence (back)', 'median'),
                                       cadence_spread=('cadence (back)', lambda v: v.std() / v.mean() * 100),
                                       step_regularity=('step regularity', 'median'),
                                       stride_regularity=('stride regularity', 'median'),
                                       jolt=('trunk jolt (g)', 'median'))
per_person['steady share %'] = 100 * walk.groupby('pid').agree.mean()
per_person['% windows at 100+ steps/min'] = 100 * steady.groupby('pid')['cadence (back)'].apply(lambda v: (v >= MODERATE_CADENCE).mean())
per_person.to_csv(RESULTS_DIR / 'stage6_har70_per_person.csv')
print('per person (steady walking windows only):')
print(per_person.round(2).to_string())

per person (steady walking windows only):
     windows  cadence  cadence_spread  step_regularity  stride_regularity  jolt  steady share %  % windows at 100+ steps/min
pid                                                                                                                         
501      103   113.89           11.62             0.85               0.79  0.31           91.15                        79.61
502       46    98.69           20.30             0.55               0.48  0.22           36.22                        36.96
503       77   110.23            5.51             0.77               0.81  0.24           76.24                        96.10
504       49   104.37            5.69             0.50               0.45  0.24           44.14                        87.76
505       16    93.02            8.69             0.65               0.50  0.10           55.17                        12.50
506      110   108.03            7.22             0.65               0.73  0.29    

## 3. How fast, how regular, how intense?

In [5]:
print(f'cadence, steady walking: median {steady["cadence (back)"].median():.0f} steps/min '
      f'(middle half {steady["cadence (back)"].quantile(.25):.0f} to {steady["cadence (back)"].quantile(.75):.0f})')
print(f'person medians range from {per_person.cadence.min():.0f} to {per_person.cadence.max():.0f} steps/min')
print(f'windows at {MODERATE_CADENCE}+ steps/min (moderate intensity): {100 * (steady["cadence (back)"] >= MODERATE_CADENCE).mean():.0f}%. '
      f'People whose median is {MODERATE_CADENCE}+: {(per_person.cadence >= MODERATE_CADENCE).sum()} of {len(per_person)}')
print(f'cadence varies within a person by {per_person.cadence_spread.median():.1f}% (median coefficient of variation)')

print('\nwalking vs shuffling (all windows, medians):')
print(win.groupby('activity')[['cadence (back)', 'step regularity', 'stride regularity', 'trunk jolt (g)']].median().round(3).to_string())

bins = [0, 90, 100, 110, 120, 200]
print('\ndoes faster walking jolt the trunk harder? (steady windows)')
print(steady.groupby(pd.cut(steady['cadence (back)'], bins), observed=True)['trunk jolt (g)'].agg(['size', 'median']).round(3).to_string())

cadence, steady walking: median 104 steps/min (middle half 96 to 111)
person medians range from 66 to 114 steps/min
windows at 100+ steps/min (moderate intensity): 63%. People whose median is 100+: 10 of 18
cadence varies within a person by 10.1% (median coefficient of variation)

walking vs shuffling (all windows, medians):
           cadence (back)  step regularity  stride regularity  trunk jolt (g)
activity                                                                     
shuffling         125.610            0.132              0.125           0.038
walking           101.807            0.593              0.532           0.181

does faster walking jolt the trunk harder? (steady windows)
                size  median
cadence (back)              
(0, 90]          235   0.115
(90, 100]        193   0.179
(100, 110]       427   0.230
(110, 120]       247   0.308
(120, 200]        70   0.317


## 4. How long are the walking bouts?

The models need 30 s windows, and Stage 4 found the HMM smooths away short bursts of activity. If very old people walk in short bouts, much of their walking could be invisible to the watch.

In [6]:
wb = all_bouts[all_bouts.activity == 'walking']
edges = [0, 10, 30, 60, 120, 300, np.inf]
names = ['under 10 s', '10–30 s', '30–60 s', '1–2 min', '2–5 min', 'over 5 min']
groups = pd.cut(wb.seconds, edges, labels=names, right=False)
bout_table = pd.DataFrame({'bouts': wb.groupby(groups, observed=False).size(),
                           '% of walking time': 100 * wb.groupby(groups, observed=False).seconds.sum() / wb.seconds.sum()})
print(bout_table.round(1).to_string())
short_share = 100 * wb.seconds[wb.seconds < 30].sum() / wb.seconds.sum()
print(f'\n{len(wb)} walking bouts, median {wb.seconds.median():.0f} s. Walking time in bouts under 30 s: {short_share:.0f}%')
print('per person, % of walking time in bouts under 30 s:',
      (100 * wb[wb.seconds < 30].groupby('pid').seconds.sum() / wb.groupby('pid').seconds.sum()).fillna(0).round(0).astype(int).to_dict())
print('\nNote: HAR70+ was recorded in a semi-structured session, so its bout lengths are only a rough guide to a real day.')

            bouts  % of walking time
seconds                             
under 10 s    481               10.2
10–30 s       188               15.1
30–60 s        79               15.4
1–2 min        63               24.3
2–5 min        31               25.5
over 5 min      5                9.5

847 walking bouts, median 8 s. Walking time in bouts under 30 s: 25%
per person, % of walking time in bouts under 30 s: {'501': 11, '502': 11, '503': 18, '504': 14, '505': 63, '506': 11, '507': 20, '508': 21, '509': 19, '510': 25, '511': 18, '512': 40, '513': 27, '514': 50, '515': 37, '516': 39, '517': 25, '518': 41}

Note: HAR70+ was recorded in a semi-structured session, so its bout lengths are only a rough guide to a real day.


## 5. Back to GOTOV: were the "light" walking windows really slower?

GOTOV's chest sensor sits on the trunk like HAR70+'s back sensor, so the same back method gives the cadence of every GOTOV walking window. Then, for each window: what cadence, and what did the models say?

(The chest sensor was worn by 29 of the 35 people. The chest and wrist clocks are synchronised by GOTOV, so the same 30 s can be cut from both.)

In [7]:
stage5 = np.load(RESULTS_DIR / 'stage5_gotov_predictions.npz')
WALKS = ['walkingSlow', 'walkingNormal', 'walkingFast']
is_walk = np.isin(stage5['activity'], WALKS)
archive = zipfile.ZipFile(GOTOV_DIR / 'Activity_Measurements.zip')
chest_files = {n.split('/')[1]: n for n in archive.namelist()
               if n.lower().endswith('-geneactiv-chest.csv') and not n.startswith('__MACOSX')}

rows = []
for p in sorted(set(stage5['pid'][is_walk]) & set(chest_files)):
    d = pd.read_csv(archive.open(chest_files[p]), usecols=['time', 'x', 'y', 'z'])
    t = d['time'].to_numpy('float64'); xyz = d[['x', 'y', 'z']].to_numpy()
    for i in np.flatnonzero(is_walk & (stage5['pid'] == p)):
        m = (t >= stage5['start'][i]) & (t < stage5['start'][i] + 30_000)
        if m.sum() < 0.9 * 30 * 85:                    # chest data missing for part of this window
            continue
        fs = (m.sum() - 1) / ((t[m][-1] - t[m][0]) / 1000)
        cad = cadence_trunk(xyz[m], fs)
        rows.append({'pid': p, 'activity': stage5['activity'][i], 'cadence': cad,
                     'step regularity': regularity(xyz[m], fs, cad)[0],
                     **{m_: CLASSES[stage5[f'pred_{m_}'][i]] for m_ in ['RF+HMM', 'CNN+HMM', 'XGB+HMM']}})
gw = pd.DataFrame(rows)
gw.to_csv(RESULTS_DIR / 'stage6_gotov_walking_cadence.csv', index=False)
print(f'{len(gw)} GOTOV walking windows with chest data, from {gw.pid.nunique()} people')
print('\ncadence by walking speed (should rise from slow to fast):')
print(gw.groupby('activity').cadence.describe()[['count', '25%', '50%', '75%']].reindex(WALKS).round(0).to_string())

460 GOTOV walking windows with chest data, from 16 people

cadence by walking speed (should rise from slow to fast):
               count    25%    50%    75%
activity                                 
walkingSlow    158.0  102.0  107.0  111.0
walkingNormal  157.0  112.0  116.0  120.0
walkingFast    145.0  120.0  125.0  128.0


In [8]:
gw['cadence band'] = pd.cut(gw.cadence, [0, 90, 100, 110, 120, 200], right=False)
called = gw.groupby('cadence band', observed=True)[['RF+HMM', 'CNN+HMM', 'XGB+HMM']].agg(lambda v: 100 * (v == 'moderate-vigorous').mean())
called.insert(0, 'windows', gw.groupby('cadence band', observed=True).size())
print('% of GOTOV walking windows called moderate-vigorous, by cadence:')
print(called.round(0).to_string())

print('\nmedian cadence of walking windows, by what XGB+HMM called them:')
print(gw.groupby('XGB+HMM').cadence.agg(['size', 'median']).round(0).to_string())

print('\nby speed: % at 100+ steps/min, and % XGB+HMM called moderate-vigorous:')
print(pd.DataFrame({'% at 100+ steps/min': gw.groupby('activity').cadence.apply(lambda v: 100 * (v >= MODERATE_CADENCE).mean()),
                    '% called moderate-vigorous': gw.groupby('activity')['XGB+HMM'].apply(lambda v: 100 * (v == 'moderate-vigorous').mean())})
      .reindex(WALKS).round(0).to_string())

print(f'\nHAR70+ (70–95) steady walking, median cadence {steady["cadence (back)"].median():.0f}; '
      f'GOTOV (60–81) normal walking {gw[gw.activity == "walkingNormal"].cadence.median():.0f}, '
      f'slow walking {gw[gw.activity == "walkingSlow"].cadence.median():.0f}')

% of GOTOV walking windows called moderate-vigorous, by cadence:
              windows  RF+HMM  CNN+HMM  XGB+HMM
cadence band                                   
[0, 90)             1     0.0      0.0      0.0
[90, 100)          29    34.0     38.0     38.0
[100, 110)        115    31.0     40.0     41.0
[110, 120)        170    32.0     34.0     42.0
[120, 200)        145    46.0     17.0     50.0

median cadence of walking windows, by what XGB+HMM called them:
                   size  median
XGB+HMM                        
light               259   114.0
moderate-vigorous   201   117.0

by speed: % at 100+ steps/min, and % XGB+HMM called moderate-vigorous:
               % at 100+ steps/min  % called moderate-vigorous
activity                                                      
walkingSlow                   84.0                        36.0
walkingNormal                 97.0                        46.0
walkingFast                  100.0                        50.0

HAR70+ (70–95) ste

## 6. If not cadence, then what? A check of the sensor's axes

Section 5 shows the models' "light" vs "moderate-vigorous" call barely follows cadence. Stage 5 (section 3) found GOTOV's wrist axes point differently from Capture-24's, which corrupts the angle features (`x_mean`, `y_mean`, `z_mean`, `pitch`, `roll`) and the CNN's raw input.

**Choosing a rotation without labels.** A sensor can only be turned, not mirrored, so there are four candidates: no change, or a half-turn around one axis (flipping the other two). For each, compare the spread of each axis's gravity reading over *all* GOTOV windows with Capture-24's (10th to 90th percentiles). No activity labels are used.

**Honesty note:** the choice uses no labels, but it was made *after* seeing Stage 5's results, so treat this as a diagnosis, not a final score. The SilverGuard watch will need the same kind of axis alignment before export.

In [9]:
import subprocess
import keras
import xgboost as xgb
from common import FEATURES, make_features, scores, smooth, to_binary

with np.load('processed/features_all.npz') as f:
    c24_axes = f['F'][:, [FEATURES.index(k) for k in ['x_mean', 'y_mean', 'z_mean']]]
# GOTOV windows, in exactly the Stage 5 order
Xg, pid_g, start_g = [], [], []
for p in sorted(np.unique(stage5['pid'])):
    with np.load(f'processed/gotov/{p}.npz') as f:
        order = np.argsort(f['start'])
        Xg.append(f['X'][order]); start_g.append(f['start'][order]); pid_g.append(np.full(len(order), p))
Xg, pid_g, start_g = np.concatenate(Xg), np.concatenate(pid_g), np.concatenate(start_g)
assert (pid_g == stage5['pid']).all() and (start_g == stage5['start']).all()

gotov_axes = Xg.mean(axis=1)                       # mean of each axis per window = the gravity direction
candidates = {'no change': [1, 1, 1], 'half-turn around x (flip y, z)': [1, -1, -1],
              'half-turn around y (flip x, z)': [-1, 1, -1], 'half-turn around z (flip x, y)': [-1, -1, 1]}
distance = {name: sum(abs(np.percentile(gotov_axes[:, k] * sign[k], q) - np.percentile(c24_axes[:, k], q))
                      for k in range(3) for q in [10, 25, 50, 75, 90]) for name, sign in candidates.items()}
print("how far GOTOV's axes are from Capture-24's (smaller = closer):")
print({k: round(v, 2) for k, v in distance.items()})
best = min(distance, key=distance.get)
print('chosen:', best)

how far GOTOV's axes are from Capture-24's (smaller = closer):
{'no change': np.float64(6.28), 'half-turn around x (flip y, z)': np.float64(4.46), 'half-turn around y (flip x, z)': np.float64(5.88), 'half-turn around z (flip x, y)': np.float64(7.55)}
chosen: half-turn around x (flip y, z)


In [10]:
Xr = Xg * np.array(candidates[best], dtype='float32')
Fr = make_features(Xr)
np.savez('processed/gotov/features_rotated.npz', F=Fr)
subprocess.run([str(Path.home() / 'anaconda3' / 'python.exe'), 'rf_predict.py', 'processed/gotov/features_rotated.npz',
                'processed/gotov/rf_probs_rotated.npy'], check=True, capture_output=True)
probs_r = {'RF': np.load('processed/gotov/rf_probs_rotated.npy'),
           'CNN': keras.models.load_model(RESULTS_DIR / 'cnn_4class.keras').predict(Xr, batch_size=256, verbose=0)}
booster = xgb.XGBClassifier(); booster.load_model(RESULTS_DIR / 'xgb_4class.json')
probs_r['XGB'] = booster.predict_proba(Fr)
hmm = np.load(RESULTS_DIR / 'hmm.npz')
y_g, act_g = stage5['y'], stage5['activity']
to3 = lambda v: np.maximum(np.asarray(v), 1) - 1
rows, by_speed = [], []
for name in ['RF', 'CNN', 'XGB']:
    for version, pr in [('as recorded', stage5[f'pred_{name}+HMM']), ('rotated', smooth(probs_r[name], pid_g, hmm['start'], hmm['transition']))]:
        rows.append({'model': f'{name}+HMM', 'axes': version, 'binary kappa': scores(to_binary(y_g), to_binary(pr), 2)['kappa'],
                     '3-class kappa': scores(to3(y_g), to3(pr), 3)['kappa'],
                     'lying called sleep %': 100 * (pr[np.isin(act_g, ['lyingDownLeft', 'lyingDownRight'])] == 0).mean()})
        by_speed.append({'model': f'{name}+HMM', 'axes': version,
                         **{a: 100 * (pr[act_g == a] == 3).mean() for a in WALKS + ['cycling']},
                         'sittingChair called light %': 100 * (pr[act_g == 'sittingChair'] == 2).mean(),
                         'standing called sedentary %': 100 * (pr[act_g == 'standing'] == 1).mean()})
rotation = pd.DataFrame(rows).set_index(['model', 'axes'])
rotation.to_csv(RESULTS_DIR / 'stage6_gotov_rotated_metrics.csv')
print(rotation.round(3).to_string())
print('\n% of each activity called moderate-vigorous, and the Stage 5 weak spots:')
print(pd.DataFrame(by_speed).set_index(['model', 'axes']).round(0).to_string())

                     binary kappa  3-class kappa  lying called sleep %
model   axes                                                          
RF+HMM  as recorded         0.883          0.748                 0.000
        rotated             0.973          0.829                 8.193
CNN+HMM as recorded         0.914          0.698                66.506
        rotated             0.920          0.718                66.747
XGB+HMM as recorded         0.895          0.729                 0.000
        rotated             0.990          0.825                18.313

% of each activity called moderate-vigorous, and the Stage 5 weak spots:
                     walkingSlow  walkingNormal  walkingFast  cycling  sittingChair called light %  standing called sedentary %
model   axes                                                                                                                   
RF+HMM  as recorded         22.0           31.0         55.0     94.0                         16.0    

## 7. What this shows

**How very old people walk (HAR70+, 70–95)**

1. **Cadence: median 104 steps/min** (middle half 96 to 111) in steady walking, so most of their walking is at or above the moderate-intensity line of about 100 (63% of steady windows; 10 of 18 people have a median of 100+). But people differ a lot: from 66 to 114 steps/min.
2. **Slower walkers are also less steady and move much less.** The four slowest (66–90 steps/min) have the lowest regularity, and trunk jolt falls from about 0.31 g at 110+ steps/min to 0.115 g below 90. For them, walking really is gentle.
3. **Shuffling has no rhythm** (regularity 0.13 vs 0.59 for walking) and very little movement (0.038 g). A wrist model would likely see it as standing or light activity.
4. **Their walking comes in short bouts:** median bout 8 s, and **25% of walking time is in bouts under 30 s** (up to 63% for one person). The models need 30 s windows, and the HMM smooths short bursts away, so this walking is largely invisible to the watch as built.
5. **The method holds up:** back and thigh, two sensors measuring two different rhythms, give the same cadence within 5 steps/min for 66% of windows (median difference 1.5). The rest are turns, starts and stops, or uneven steps.

**Why GOTOV's walking was called "light" (Stage 5)**

6. **Not because it was slow.** GOTOV's walking was brisk: median 107 (slow), 116 (normal) and 125 (fast) steps/min, with 97% of "normal" windows at 100+. And the models' calls barely followed cadence: windows XGBoost called light had a median of 114 steps/min, against 117 for moderate-vigorous.
7. **The real cause was the sensor's axes.** Turning GOTOV's wrist data half a turn around its x axis (chosen without labels, by matching the spread of each axis to Capture-24's) fixes most of it. Still vs moving kappa rises to 0.97 (forest) and 0.99 (XGBoost), and normal walking is called moderate-vigorous 91% of the time instead of 31–40%. The desk-sitting and standing errors from Stage 5 mostly disappear too.
8. **What the rotation doesn't fix:** lying awake is now called sleep more often by the tree models (8–18%, from 0%), and the CNN still calls it sleep 67% of the time.
9. **GOTOV's "walking slow" is probably moderate, not light.** 84% of those windows are at 100+ steps/min, and after rotation the tree models call 71–80% of them moderate-vigorous. Section 1 of Stage 5 marked this mapping as a judgement call; cadence says it should go the other way.

**What it means for SilverGuard**

- **Lining up the watch's axes with Capture-24's is essential**, not optional. One flipped axis turned a 0.99 still-vs-moving model into a 0.90 one and hid most moderate walking
- **Short walking bouts** need a design that can see them (shorter windows, or an HMM that doesn't erase short bursts) if the watch is meant to count very old people's activity
- **Caveat:** the rotation was chosen after seeing Stage 5's results, even though no labels were used. It needs confirming on the watch's own sensor

In [11]:
# every claim above, checked against the numbers
assert round(steady['cadence (back)'].median()) == 104 and round(steady['cadence (back)'].quantile(.25)) == 96
assert round(steady['cadence (back)'].quantile(.75)) == 111                                            # 1
assert round(100 * (steady['cadence (back)'] >= 100).mean()) == 63 and (per_person.cadence >= 100).sum() == 10
assert round(per_person.cadence.min()) == 66 and round(per_person.cadence.max()) == 114
slow4 = per_person.nsmallest(4, 'cadence')                                                              # 2
assert slow4.cadence.max() < 91 and slow4.step_regularity.max() < per_person.step_regularity.median()
jolt = steady.groupby(pd.cut(steady['cadence (back)'], bins), observed=True)['trunk jolt (g)'].median()
assert round(jolt.iloc[0], 3) == 0.115 and jolt.iloc[-2:].min() > 0.3
sh, wk = win[win.activity == 'shuffling'], win[win.activity == 'walking']                               # 3
assert round(sh['step regularity'].median(), 2) == 0.13 and round(wk['step regularity'].median(), 2) == 0.59
assert round(sh['trunk jolt (g)'].median(), 3) == 0.038
assert round(wb.seconds.median()) == 8 and round(short_share) == 25                                    # 4
assert round(100 * (gap <= 5).mean()) == 66 and round(gap.median(), 1) == 1.5                          # 5
med = gw.groupby('activity').cadence.median()                                                           # 6
assert [round(med[a]) for a in WALKS] == [107, 116, 125]
assert round(100 * (gw[gw.activity == 'walkingNormal'].cadence >= 100).mean()) == 97
assert gw.groupby('XGB+HMM').cadence.median().round().to_dict() == {'light': 114, 'moderate-vigorous': 117}
assert best == 'half-turn around x (flip y, z)'                                                         # 7
r = rotation['binary kappa']
assert round(r['RF+HMM', 'rotated'], 2) == 0.97 and round(r['XGB+HMM', 'rotated'], 2) == 0.99
sp = pd.DataFrame(by_speed).set_index(['model', 'axes'])
assert all(round(sp.loc[(m, 'rotated'), 'walkingNormal']) == 91 for m in ['RF+HMM', 'XGB+HMM'])
assert 31 <= sp.loc[('RF+HMM', 'as recorded'), 'walkingNormal'] and sp.loc[('XGB+HMM', 'as recorded'), 'walkingNormal'] <= 40.5
ls = rotation['lying called sleep %']                                                                   # 8
assert 8 <= ls['RF+HMM', 'rotated'] < 9 and 18 <= ls['XGB+HMM', 'rotated'] < 19 and round(ls['CNN+HMM', 'rotated']) == 67
assert round(100 * (gw[gw.activity == 'walkingSlow'].cadence >= 100).mean()) == 84                     # 9
assert all(70 <= sp.loc[(m, 'rotated'), 'walkingSlow'] <= 80.5 for m in ['RF+HMM', 'XGB+HMM'])
print('every claim above matches the numbers')

every claim above matches the numbers
